In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
data_previous = pd.read_csv('/content/drive/My Drive/data/issuer_data.csv')

In [ ]:
unique_list = data_previous["corp_tkr"].unique().tolist()

In [ ]:
len(unique_list)

In [ ]:
data_previous.dt #2020-12-31, 2025-02-14

In [ ]:
! pip uninstall -y polygon
! pip install -U polygon-api-client


In [ ]:
pd.to_datetime(1698033600000, unit="ms", utc=True)

In [ ]:
# #polygen way of dragging data
# from polygon import RESTClient

# client = RESTClient("iFbqo8J_h417SUepWnMfeimKvy2Oqca9")
# # pip install -U polygon-api-client pandas
# from polygon import RESTClient
# import pandas as pd

# API_KEY = "iFbqo8J_h417SUepWnMfeimKvy2Oqca9"

# def _val(obj, key):
#     if isinstance(obj, dict):
#         return obj.get(key)
#     return getattr(obj, key, None)

# def fetch_aggs_to_df(
#     tickers,
#     start="2020-12-31",
#     end="2025-02-14",
#     timespan="day",
#     multiplier=1,
#     adjusted=True,
#     sort="asc",
#     limit=5000,
#     api_key=None,
# ):
#     client = RESTClient(api_key)
#     rows = []
#     for tk in tickers:
#         aggs_iter = client.list_aggs(
#             tk,
#             multiplier,
#             timespan,
#             start,
#             end,
#             adjusted=adjusted,   # 布尔，不要写成 "true"
#             sort=sort,
#             limit=limit,
#         )
#         for a in aggs_iter:
#             rows.append({
#                 "corp_tkr": tk,
#                 "dt": pd.to_datetime(_val(a, "t"), unit="ms", utc=True),
#                 "open": _val(a, "open"),
#                 "low": _val(a, "low"),
#                 "high": _val(a, "high"),
#                 "volume": _val(a, "volume"),
#                 "close": _val(a, "close"),
#                 "vwap": _val(a, "vwap"),
#                 "transactions": _val(a, "transactions"),
#             })
#     df = pd.DataFrame(rows)
#     if not df.empty:
#         df = df.sort_values(["ticker", "time"]).reset_index(drop=True)
#     return df

# tickers = ["AAPL", "MSFT", "NVDA"]
# df = fetch_aggs_to_df(
#     tickers,
#     start="2020-12-31",
#     end="2025-02-14",
#     api_key=API_KEY,
# )
# print(df.head())
# print(df.tail())


In [ ]:
! pip install yfinance

In [ ]:
# # pip install yfinance pandas
# import yfinance as yf
# import pandas as pd

# def _to_utc(s):
#     if s.dt.tz is None:
#         return s.dt.tz_localize("UTC")
#     return s.dt.tz_convert("UTC")

# def fetch_yf_to_df(
#     tickers,
#     start=None,
#     end=None,
#     period=None,
#     interval="1d",
#     auto_adjust=False,
# ):
#     rows = []
#     for tk in tickers:
#         hist = yf.Ticker(tk).history(start=start, end=end, period=period,
#                                      interval=interval, auto_adjust=auto_adjust)
#         if hist.empty:
#             continue
#         df = hist.reset_index()
#         df.rename(columns={
#             "Date": "dt",
#             "Open": "open",
#             "Low": "low",
#             "High": "high",
#             "Close": "close",
#             "Volume": "volume",
#         }, inplace=True)
#         df["dt"] = _to_utc(pd.to_datetime(df["dt"]))
#         df["corp_tkr"] = tk
#         rows.append(df[["corp_tkr", "dt", "open", "low", "high", "volume", "close"]])

#     if not rows:
#         return pd.DataFrame(columns=["corp_tkr","dt","open","low","high","volume","close","vwap","transactions"])

#     out = (pd.concat(rows, ignore_index=True)
#              .sort_values(["corp_tkr", "dt"])
#              .reset_index(drop=True))
#     return out

# tickers = unique_list
# df = fetch_yf_to_df(tickers, start="2020-12-31", end="2025-02-14", interval="1d", auto_adjust=False)

# # df = fetch_yf_to_df(tickers, period="5y", interval="1d", auto_adjust=False)

# print(df.head())
# print(df.tail())
# print("rows:", len(df))


In [ ]:
# pip install pandas
import pandas as pd
import re

# mapping
# Old Code → New Code (Mergers/Renames)
SYMBOL_MAP = {
    "ABC": "COR",
    "VIAC": "PARA",
    "BHI": "BKR",
    "FLIR": "TDY",
    "FISV": "FI",
    "PKI": "RVTY",
    "PXD": "XOM",
    "RPAI": "KIM",
    "NGLS": "TRGP",
    "DCP": "PSX",
    "CBG": "CBRE",
    "TWC": "CHTR",
    "AET": "CVS",
    "VMW": "AVGO",
    "BRK": "BRK-B",
    "TOYOTA": "TM",
    "HNDA": "HMC",
    'CHCOCH':"LNG",
    'FTSCN':'FTS',
    'OLHSNL':'HUM',
    'NGGLN':"NGG",
    'SANUSA':'SAN',
    'BNSF':"BRK-B"
}

NON_TICKER_LIKE = {
    "ASCHEA","BANNER","BSHSI","CATMED","CHCOCH","EMORYU","FTSCN","ONCRTX","PSJHOG",
    "RUSHOB","STNFHC","SUTHEA","YALUNI","BSWHLD","OLHSNL","PIEDGA","CEDARS","JHUNIV",
    "NGGLN","UPMCHS","SANUSA","BNSF","COLUNV","NOVANT"}



In [ ]:
# pip install yfinance pandas
"""
yfinance_fetch_pipeline
Purpose
-------
Build a robust, standalone pipeline to:
1) Normalize "corporate" tickers from an upstream dataset,
2) Filter out known non-ticker-like identifiers,
3) Map legacy / renamed tickers to currently tradable Yahoo Finance symbols,
4) Download historical OHLCV time series from Yahoo Finance via yfinance,
5) Return a long-format DataFrame while preserving the original corporate ticker
   and the mapped (renamed) tradable ticker.
"""

import yfinance as yf
import pandas as pd
import re

SYMBOL_MAP = {
    "ABC": "COR",
    "VIAC": "PARA",
    "BHI": "BKR",
    "FLIR": "TDY",
    "FISV": "FI",
    "PKI": "RVTY",
    "PXD": "XOM",
    "RPAI": "KIM",
    "NGLS": "TRGP",
    "DCP": "PSX",
    "CBG": "CBRE",
    "TWC": "CHTR",
    "AET": "CVS",
    "VMW": "AVGO",
    "BRK": "BRK-B",
    "TOYOTA": "TM",
    "HNDA": "HMC",
    "CHCOCH":"LNG",
    "FTSCN":"FTS",
    "OLHSNL":"HUM",
    "NGGLN":"NGG",
    "SANUSA":"SAN",
    "BNSF":"BRK-B",
    "DISCA": "WBD",
    "ENBL": "ET",
    "SEP": "ENB",
    "CDWC": "CDW"
}

NON_TICKER_LIKE = {
    "ASCHEA","BANNER","BSHSI","CATMED","CHCOCH","EMORYU","FTSCN","ONCRTX","PSJHOG",
    "RUSHOB","STNFHC","SUTHEA","YALUNI","BSWHLD","OLHSNL","PIEDGA","CEDARS","JHUNIV",
    "NGGLN","UPMCHS","SANUSA","BNSF","COLUNV","NOVANT","BRKHEC","SPLLLC","OHCMED","HARVRD","PSD"
}

def _norm(sym: str) -> str:
    """
    Normalize a ticker-like string into a Yahoo Finance friendly format
    Operations
    ----------
    - Strip whitespace and leading '$'
    - Upper-case the symbol
    - Convert single-class dot tickers (e.g., "BRK.B") to Yahoo format ("BRK-B")
      only when pattern matches: LETTERS + '.' + SINGLE LETTER
    """
    if not isinstance(sym, str): return ""
    s = sym.strip().upper().lstrip("$")
    if re.match(r'^[A-Z]+\.([A-Z])$', s):
        s = s.replace('.', '-')
    return s

def _to_utc(s):
    if s.dt.tz is None:
        return s.dt.tz_localize("UTC")
    return s.dt.tz_convert("UTC")

def prepare_fetch_table(unique_list):
    """
    Prepare a symbol-fetch table from a raw list of identifiers.

    Output columns：
    - corp_tkr: original identifier (kept exactly as provided)
    - renamed_ticker: mapped tradable ticker (if available in SYMBOL_MAP), else NaN
    - fetch_symbol: the ticker actually used for yfinance download:
        * renamed_ticker if mapped
        * else normalized corp_tkr

    Filtering：
    Any normalized identifier in NON_TICKER_LIKE will be removed.
    """
    df = pd.DataFrame({"corp_tkr": unique_list})
    df["corp_tkr_norm"] = df["corp_tkr"].astype(str).map(_norm)
    df = df[~df["corp_tkr_norm"].isin(NON_TICKER_LIKE)].copy()
    df["renamed_ticker"] = df["corp_tkr_norm"].map(lambda x: SYMBOL_MAP.get(x))
    df["fetch_symbol"] = df["renamed_ticker"].fillna(df["corp_tkr_norm"])
    df = df.drop_duplicates(subset=["corp_tkr"]).reset_index(drop=True)
    return df[["corp_tkr", "renamed_ticker", "fetch_symbol"]]

def fetch_yf_to_df(prep_df, start=None, end=None, period=None, interval="1d", auto_adjust=False):
    """
    Download historical OHLCV data from Yahoo Finance using a prepared fetch table.

    For each row in prep_df:
    - Uses fetch_symbol to download data via yfinance,
    - Converts date/time to UTC,
    - Attaches corp_tkr (original identifier) and renamed_ticker (if applicable),
    - Collects all results into a single long-format DataFrame.
    """
    rows = []
    for _, row in prep_df.iterrows():
        orig = row["corp_tkr"]
        renamed = row["renamed_ticker"]
        sym = row["fetch_symbol"]

        try:
            hist = yf.Ticker(sym).history(start=start, end=end, period=period,
                                          interval=interval, auto_adjust=auto_adjust)
        except Exception:
            continue

        if hist.empty:
            continue

        df = hist.reset_index().rename(columns={
            "Date":"dt","Open":"open","Low":"low","High":"high","Close":"close","Volume":"volume"
        })
        df["dt"] = _to_utc(pd.to_datetime(df["dt"]))
        df["corp_tkr"] = orig
        df["renamed_ticker"] = renamed
        rows.append(df[["corp_tkr","renamed_ticker","dt","open","low","high","volume","close"]])

    if not rows:
        return pd.DataFrame(columns=["corp_tkr","renamed_ticker","dt","open","low","high","volume","close"])

    out = (pd.concat(rows, ignore_index=True)
             .sort_values(["corp_tkr", "dt"])
             .reset_index(drop=True))
    return out

# unique_list = [...]

prep = prepare_fetch_table(unique_list)
print("Items to be downloaded (excluding non-ticker-like entries):")
print(prep.head(20))

df = fetch_yf_to_df(prep, start="2020-12-31", end="2025-02-14", interval="1d", auto_adjust=False)
print(df.head(), "\nrows:", len(df))


In [ ]:
df["dt"] = df["dt"].dt.date

In [ ]:
import pandas as pd

dp = data_previous.copy()
dp["corp_tkr"] = dp["corp_tkr"].astype(str).str.upper()
dp["dt"] = pd.to_datetime(dp["dt"]).dt.date
dp["_order"] = range(len(dp))

df2 = df.copy()
df2["corp_tkr"] = df2["corp_tkr"].astype(str).str.upper()
df2["dt"] = pd.to_datetime(df2["dt"]).dt.date
df2 = df2.drop_duplicates(subset=["corp_tkr","dt"])


non_key_cols = [c for c in df2.columns if c not in ["corp_tkr","dt"]]
df_slim = df2[["corp_tkr","dt"] + non_key_cols]

merged = dp.merge(df_slim, how="left", on=["corp_tkr","dt"])
merged = merged.sort_values("_order").drop(columns="_order").reset_index(drop=True)
print(merged.head())


In [ ]:
df_new_senti=pd.read_csv('/content/drive/My Drive/data/merged_data_v3.csv')

In [ ]:
import pandas as pd
import numpy as np
# --------------------- Prepare sentiment/event dataframe ---------------------
# Create a working copy to avoid mutating the original input
df = df_new_senti.copy()
df["corp_tkr"] = df["corp_tkr"].astype(str).str.upper()
df["dt"] = pd.to_datetime(df["dt"])

# --------------------- Define event category columns ---------------------
# List of event-related one-hot columns
event_cols = [
    "Analyst & Research Events","Bankruptcy / Restructuring",
    "Capital-Markets Deals","Corporate Actions","Dividends","ESG",
    "Guidance","Index Membership","Leadership","Legal","M&A",
    "Operations & Business Changes","Product Events","Regulatory",
    "Strategic & Financing Intent"
]
key_cols = ["corp_tkr","dt"]
other_cols = [c for c in df.columns if c not in key_cols + event_cols]

# --------------------- Normalize event indicators ---------------------
# Convert event columns to strict binary indicators (0/1)
# - Fill missing values with 0
# - Cast to integer
# - Clip to [0, 1] to guard against unexpected values (>1)
df[event_cols] = df[event_cols].fillna(0).astype(int).clip(0, 1)

# --------------------- Aggregate to daily stock-level ---------------------
# Aggregation logic:
# - Event columns: take max (whether the event occurred at least once that day)
# - Other columns: take first (assumed invariant within the same stock-day)
agg_dict = {c: "max" for c in event_cols}
agg_dict.update({c: "first" for c in other_cols})

daily_onehot = (
    df.groupby(key_cols, as_index=False)
      .agg(agg_dict)
      .sort_values(key_cols)
      .reset_index(drop=True)
)

In [ ]:
daily_onehot.columns

In [ ]:
df_new_senti[['dt', 'fs_entity_id', 'capiq_id', 'corp_tkr',
       'Analyst & Research Events', 'Bankruptcy / Restructuring',
       'Capital-Markets Deals', 'Corporate Actions', 'Dividends', 'ESG',
       'Guidance', 'Index Membership', 'Leadership', 'Legal', 'M&A',
       'Operations & Business Changes', 'Product Events', 'Regulatory',
       'Strategic & Financing Intent']]

In [ ]:
# --------------------- Event count construction ---------------------
# Define event-related columns (excluding primary key or identifier columns)
event_cols = ['Analyst & Research Events', 'Bankruptcy / Restructuring',
       'Capital-Markets Deals', 'Corporate Actions', 'Dividends', 'ESG',
       'Guidance', 'Index Membership', 'Leadership', 'Legal', 'M&A',
       'Operations & Business Changes', 'Product Events', 'Regulatory',
       'Strategic & Financing Intent']

# Compute the number of active events per observation
# Each event column is assumed to be a binary indicator (0/1),
# so summing across columns gives the total event count for that row.
df_new_senti["event_count"] = df_new_senti[event_cols].sum(axis=1)
max_events = df_new_senti["event_count"]

max_events

In [ ]:
# --------------------- Event-based factor preparation (equity, 0/5/10d horizons) ---------------------
# This block constructs event-count–based factors at the stock-day level.

# Create a working copy to avoid mutating the original dataframe
df = df_new_senti.copy()
df["corp_tkr"] = df["corp_tkr"].astype(str).str.upper()
df["dt"] = pd.to_datetime(df["dt"])

event_cols = ['Analyst & Research Events','Bankruptcy / Restructuring',
    'Capital-Markets Deals','Corporate Actions','Dividends','ESG',
    'Guidance','Index Membership','Leadership','Legal','M&A',
    'Operations & Business Changes','Product Events','Regulatory',
    'Strategic & Financing Intent']

# --------------------- Row-level event count ---------------------
# Count how many event categories are active in each row.
# Even if each row corresponds to a single event, this operation is robust
# to cases where multiple event flags are present.
df["event_count"] = df[event_cols].sum(axis=1)

# --------------------- Aggregate to stock-day level ---------------------
# Collapse multiple rows within the same (corp_tkr, dt) into a single observation.
# Event counts are summed to reflect total event intensity on that day.
daily = (df.groupby(["corp_tkr","dt"], as_index=False)
           .agg(event_count=("event_count","sum"))
           .sort_values(["corp_tkr","dt"]))

In [ ]:
daily

In [ ]:
# --------------------- Sector-level aggregation factors (equity, 0/5/10 day windows) ---------------------
# This block constructs sector-level event intensity factors by aggregating
# stock-level event counts to different industry hierarchies and applying
# rolling-window accumulation.
import pandas as pd
import numpy as np

sector_levels = ["ml_industry_lvl_2",'ml_industry_lvl_3', 'ml_industry_lvl_4']
windows = (5, 10)
include_today = True

def sector_roll(df, level_col, include_today=True, windows=(5,10)):
    """
    Aggregate stock-level event counts to the sector level and compute
    rolling event intensity features.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe containing at least:
        - level_col : industry classification column
        - dt : trading date
        - event_count : stock-level event count
    level_col : str
        Industry hierarchy column used for aggregation.
    include_today : bool, default True
        Whether to include current-day events in the rolling window.
        If False, event counts are shifted by one day to avoid look-ahead bias.
    windows : tuple of int
        Rolling window lengths (in calendar days).

    Returns
    -------
    pd.DataFrame
        Sector-level rolling event intensity features indexed by (level_col, dt).
    """
    sec_daily = (df.groupby([level_col, "dt"], as_index=False)
                   .agg(sec_events=("event_count","sum"))
                   .sort_values([level_col,"dt"]))

    def _roll(g):
        g = g.sort_values("dt").copy()
        if not include_today:
            g["sec_events_shift"] = g["sec_events"].shift(1)
            base_col = "sec_events_shift"
        else:
            base_col = "sec_events"

        out = g[[level_col, "dt"]].copy()
        out[f"{level_col}_roll0D"] = g[base_col]

        for w in windows:
            out[f"{level_col}_roll{w}D"] = (
                g.rolling(window=f"{w}D", on="dt")[base_col].sum()
            )
        return out

    return sec_daily.groupby(level_col, group_keys=False).apply(_roll)


sector_roll_dict = {lvl: sector_roll(df, lvl, include_today, windows)
                    for lvl in sector_levels}

base = df[["corp_tkr","dt"] + sector_levels].drop_duplicates()

for lvl in sector_levels:
    roll_df = sector_roll_dict[lvl]
    base = base.merge(roll_df, on=[lvl,"dt"], how="left")
# ['corp_tkr','dt','ml_industry_lvl_1','ml_industry_lvl_2',
#  'ml_industry_lvl_1_roll0D','ml_industry_lvl_1_roll5D','ml_industry_lvl_1_roll10D',
#  'ml_industry_lvl_2_roll0D','ml_industry_lvl_2_roll5D','ml_industry_lvl_2_roll10D']


In [ ]:
def add_time_roll(g):
    """
    Compute time-series rolling event intensity for a single stock.

    This function operates on one stock's daily event data and constructs
    rolling sums of event counts over fixed calendar windows (e.g. 5D, 10D),
    capturing short-term accumulation of event activity.
    """
    s = g.set_index("dt")["event_count"].sort_index()

    g["roll5D_events_count"]  = s.rolling("5D").sum().values
    g["roll10D_events_count"] = s.rolling("10D").sum().values
    return g

daily_roll = daily.groupby("corp_tkr", group_keys=False).apply(add_time_roll)
daily_roll

In [ ]:
merged_all = merged.merge(
    base[['corp_tkr', 'dt', 'ml_industry_lvl_2_roll0D',
       'ml_industry_lvl_2_roll5D', 'ml_industry_lvl_2_roll10D',
       'ml_industry_lvl_3_roll0D', 'ml_industry_lvl_3_roll5D',
       'ml_industry_lvl_3_roll10D', 'ml_industry_lvl_4_roll0D',
       'ml_industry_lvl_4_roll5D', 'ml_industry_lvl_4_roll10D']],
    how="left",
    on=["corp_tkr","dt"]
)
merged_all=merged_all.merge(
    daily_onehot[['corp_tkr', 'dt', 'Analyst & Research Events',
       'Bankruptcy / Restructuring', 'Capital-Markets Deals',
       'Corporate Actions', 'Dividends', 'ESG', 'Guidance', 'Index Membership',
       'Leadership', 'Legal', 'M&A', 'Operations & Business Changes',
       'Product Events', 'Regulatory', 'Strategic & Financing Intent']],
    how="left",
    on=["corp_tkr","dt"]
)
merged_all=merged_all.merge(
    daily_roll[['corp_tkr', 'dt', 'event_count', 'roll5D_events_count',
       'roll10D_events_count']],
    how="left",
    on=["corp_tkr","dt"]
)

In [ ]:
cols=['Analyst & Research Events',
       'Bankruptcy / Restructuring', 'Capital-Markets Deals',
       'Corporate Actions', 'Dividends', 'ESG', 'Guidance', 'Index Membership',
       'Leadership', 'Legal', 'M&A', 'Operations & Business Changes',
       'Product Events', 'Regulatory', 'Strategic & Financing Intent','ml_industry_lvl_2_roll0D',
       'ml_industry_lvl_2_roll5D', 'ml_industry_lvl_2_roll10D',
       'ml_industry_lvl_3_roll0D', 'ml_industry_lvl_3_roll5D',
       'ml_industry_lvl_3_roll10D', 'ml_industry_lvl_4_roll0D',
       'ml_industry_lvl_4_roll5D', 'ml_industry_lvl_4_roll10D','event_count', 'roll5D_events_count',
       'roll10D_events_count']


merged_all[cols] = merged_all[cols].fillna(0)

In [ ]:
# --------------------- PCA-based dimensionality reduction (Edition 1) ---------------------
# This block applies Principal Component Analysis (PCA) to a selected feature subset
# in order to reduce dimensionality while preserving the majority of variance.

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

pca_dt=merged_all[cols]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(pca_dt)

pca = PCA(n_components=25, random_state=42)
X_pca = pca.fit_transform(X_scaled)


pca_cols = [f"PCA{i+1}" for i in range(pca.n_components_)]
pca_df = pd.DataFrame(X_pca, columns=pca_cols, index=merged_all.index)

merged_pca = pd.concat([merged_all, pca_df], axis=1)

explained_var = pca.explained_variance_ratio_

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# explained_var = pca.explained_variance_ratio_
n_components = len(explained_var)

plt.figure(figsize=(8,5))
plt.plot(np.arange(1, n_components + 1), explained_var, 'o-', label='Individual explained variance')
plt.plot(np.arange(1, n_components + 1), np.cumsum(explained_var), 's--', label='Cumulative explained variance')
plt.xlabel('Number of Components')
plt.ylabel('Explained Variance Ratio')
plt.title('Scree Plot (PCA)')
plt.xticks(np.arange(1, n_components + 1))
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# --------------------- PCA-based feature compression (Edition 1) ---------------------
# This block applies Principal Component Analysis (PCA) to reduce the dimensionality
# of a selected feature subset while preserving the dominant variance structure.

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

pca_dt=merged_all[cols]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(pca_dt)


pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# --------------------- Append PCA components to original dataset ---------------------
# Name the principal components as PCA1, PCA2, and PCA3,
# and align them with the original index before merging.
pca_cols = [f'PCA{i}' for i in [1,2,3]]
merged_all[pca_cols] = pd.DataFrame(X_pca, index=merged_all.index)


In [ ]:
merged_all

In [ ]:
merged_all.to_csv('/content/drive/My Drive/new_data/issuer_data_textual.csv')